# 6.26 Mixture-of-Experts layers

A router sends each token to a small set of experts, expanding capacity without using every parameter every time.

Sparse routing makes only a few experts active per example while retaining multiple specialized parameter sets. Save a copy to Drive to edit.

In [ ]:

import math
import time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

np.random.seed(42)


def clf_digits_ladder():
    """D1 XOR -> D2 blobs -> D3 noisy moons -> D4 digits -> D5 noisy digits."""
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def pad_to_64(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 64:
        return X.copy()
    if X.shape[1] > 64:
        return X[:, :64].copy()
    out = np.zeros((X.shape[0], 64), dtype=float)
    out[:, :X.shape[1]] = X
    return out


def split_scale(X, y):
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te




def softmax_logits(X, W):
    logits = X @ W
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def train_small_mlp_loss(X, y, random_state=0, max_iter=90):
    x_tr, x_te, y_tr, y_te = split_scale(X, y)
    clf = MLPClassifier(hidden_layer_sizes=(16,), activation="relu", solver="adam", alpha=0.001, max_iter=max_iter, random_state=random_state)
    clf.fit(x_tr, y_tr)
    proba = clf.predict_proba(x_te)
    loss = log_loss(y_te, proba, labels=clf.classes_)
    acc = accuracy_score(y_te, clf.predict(x_te))
    return float(loss), float(acc), clf


def train_small_mlp_acc(X, y, random_state=0, max_iter=90):
    loss, acc, clf = train_small_mlp_loss(X, y, random_state=random_state, max_iter=max_iter)
    return acc, loss, clf


def preview_ladder(rungs):
    for rung, (name, X, y) in enumerate(rungs, 1):
        classes = np.unique(y)
        print(f"D{rung}: {name:36s} X={X.shape} classes={len(classes)} sample_y={classes[:5].tolist()}")


def plot_metric_table(rows, metric_name):
    print(f"{'rung':<4} {'dataset':<36} {metric_name:>10}")
    for row in rows:
        print(f"{row['rung']:<4} {row['name']:<36} {row['metric']:10.4f}")


def plot_summary(rows, metric_name, ylabel):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    names = [row["rung"] for row in rows]
    values = [row["metric"] for row in rows]
    axes[0].bar(names, values, color="steelblue")
    axes[0].set_title("Metric by ladder rung")
    axes[0].set_ylabel(ylabel)
    axes[1].plot(names, values, marker="o")
    axes[1].set_title("D1→D5 trend")
    axes[1].set_ylabel(ylabel)
    fig.tight_layout()
    plt.show()


def show_artifacts(rungs, title):
    fig, axes = plt.subplots(1, len(rungs), figsize=(13, 2.6))
    for ax, (name, X, y) in zip(axes, rungs):
        if X.shape[1] == 64:
            ax.imshow(X[0].reshape(8, 8), cmap="gray")
            ax.set_title(name.split()[0])
        else:
            ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="tab10")
            ax.set_title(name.split()[0])
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


## The concept, built once

A TopK MoE computes $y=\sum_{e\in TopK}p_eE_e(x)$. The lesson scalar pass gives score $2.35$ and update $2.0-0.06\cdot1.95=1.883$.

In [ ]:

def softmax(v):
    z = v - np.max(v)
    e = np.exp(z)
    return e / e.sum()


def moe_layer(x, router_w, expert_w, top_k=2):
    probs = softmax(router_w @ x)
    keep = np.argsort(probs)[-top_k:]
    kept_probs = probs[keep] / probs[keep].sum()
    outputs = expert_w[keep] @ x
    return float(np.sum(kept_probs * outputs)), keep, kept_probs

x = np.array([1.0, 2.0])
router_w = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.5]])
expert_w = np.array([[1.0, 0.0], [0.0, 2.0], [1.0, 1.0]])
out, keep, kept_probs = moe_layer(x, router_w, expert_w, top_k=2)
score = 1.2 * 1.5 + -0.1 * -0.5 + 0.5
updated = 2.0 - 0.06 * 1.95
print("TopK experts", keep)
print("TopK probabilities", kept_probs)
print("mixture output", out)
assert len(keep) == 2
assert np.isclose(score, 2.35)
assert np.isclose(updated, 1.883)


Experts are real classifiers trained on router-defined regions; inference combines the TopK expert probabilities.

In [ ]:

def moe_predict(X_train, y_train, X_test, n_experts=3, top_k=2, balanced=True):
    n_experts = min(n_experts, max(1, len(np.unique(y_train))))
    km = KMeans(n_clusters=n_experts, random_state=26, n_init=5)
    clusters = km.fit_predict(X_train)
    experts = []
    global_clf = LogisticRegression(max_iter=1200)
    global_clf.fit(X_train, y_train)
    for expert in range(n_experts):
        idx = clusters == expert
        if idx.sum() < len(np.unique(y_train)) or len(np.unique(y_train[idx])) < 2:
            experts.append(global_clf)
        else:
            clf = LogisticRegression(max_iter=1200)
            clf.fit(X_train[idx], y_train[idx])
            experts.append(clf)
    distances = np.linalg.norm(X_test[:, None, :] - km.cluster_centers_[None, :, :], axis=2)
    router = softmax_logits(-distances, np.eye(n_experts))
    if not balanced:
        router[:, 0] = router[:, 0] + 5.0
        router = router / router.sum(axis=1, keepdims=True)
    classes = global_clf.classes_
    final = np.zeros((len(X_test), len(classes)))
    counts = np.zeros(n_experts, dtype=int)
    for i in range(len(X_test)):
        keep = np.argsort(router[i])[-top_k:]
        weights = router[i, keep] / router[i, keep].sum()
        counts[keep] = counts[keep] + 1
        for weight, expert in zip(weights, keep):
            proba = experts[expert].predict_proba(X_test[i:i + 1])[0]
            aligned = np.zeros(len(classes))
            for j, cls in enumerate(experts[expert].classes_):
                aligned[np.where(classes == cls)[0][0]] = proba[j]
            final[i] = final[i] + weight * aligned
    return classes[np.argmax(final, axis=1)], counts


def train_moe_accuracy(X, y, balanced=True):
    X64 = pad_to_64(X)
    x_tr, x_te, y_tr, y_te = split_scale(X64, y)
    preds, counts = moe_predict(x_tr, y_tr, x_te, n_experts=4, top_k=2, balanced=balanced)
    return float(accuracy_score(y_te, preds)), counts


## The dataset ladder

The same code is run from a four-point XOR problem through real 8×8 digit images and a noisy shifted D5 variant.

In [ ]:
rungs = clf_digits_ladder()
preview_ladder(rungs)
show_artifacts(rungs, 'Ladder preview')

## Run the SAME method across D1–D5

In [ ]:

rows = []
for idx, (name, X, y) in enumerate(rungs, 1):
    acc, counts = train_moe_accuracy(X, y, balanced=True)
    rows.append({"rung": f"D{idx}", "name": name, "metric": acc, "counts": counts})
plot_metric_table(rows, "accuracy")
for row in rows:
    print(row["rung"], "expert loads", row["counts"].tolist())


## Results visualization

In [ ]:
show_artifacts(rungs, 'MoE artifacts')
plot_summary(rows, 'accuracy', 'held-out accuracy')

## Pitfall on D5: router collapse

In [ ]:

name, X, y = rungs[-1]
collapsed_acc, collapsed_counts = train_moe_accuracy(X, y, balanced=False)
balanced_acc, balanced_counts = train_moe_accuracy(X, y, balanced=True)
print("collapsed accuracy", collapsed_acc, "loads", collapsed_counts.tolist())
print("balanced accuracy", balanced_acc, "loads", balanced_counts.tolist())
assert balanced_counts.min() > 0
assert balanced_acc >= collapsed_acc - 0.10


## Evaluate it + Practice

- Metric: held-out accuracy; compare it with a majority-class or plain logistic baseline.
- Sanity check: D1 should be hand-checkable before trusting D5.
- Ablation: turn off the key idea and the reported metric should get worse or the pitfall should reappear.
- Failure signals: unstable losses, collapsed routing, inflated gradient error, or a D5 result that ignores label noise.

Practice 1: change one hyperparameter and rerun the D1 assert before touching D5.

Practice 2: add a baseline row to the ladder table and explain the largest gap.

Practice 3: make the D5 pitfall more severe, then show the same fix still helps.